# Cross correlation trial

We try to see if the correlation between shots is a good metric to find out the dirty profiles

In [ ]:
# Generic imports
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

import torch
import numpy as np

from dataset import AEDataModule
from model import AutoEncoder

import seaborn as sns
%matplotlib widget
import matplotlib.pyplot as plt

from views import GAIN_PIDS as gain_list

In [ ]:
# Set the theme of the plots through seaborn
sns.set_theme(context='notebook', style='whitegrid', palette='muted', font='sans-serif', font_scale=1.2, color_codes=True, rc=None)

In [ ]:
# Load the model and the dataset
v_num = 0
model_path = f"/home/IPP-HGW/orluca/devel/classificator/W7-X_QXT/AE360/version_{v_num}/best_model_.ckpt"
model = AutoEncoder.load_from_checkpoint(model_path)
model.eval() # Set the model to evaluation mode

data_dir = '../data'
filename = '20251028_all_data.npz'
data_module = AEDataModule(data_dir, filename, batch_size=-1, normalization_strategy='minmax')
data_module.setup()
# data_module.get_pids(gain_list)
# data_module.cut_time_intervals([(4,6)], '4to6_full.npz')
# data_module.setup()


In [ ]:
from scipy.signal import correlate, find_peaks

model.eval()

index = 0
x = data_module.profiles[index]
y = model(x)

result = correlate(x.detach().numpy(), y.detach().numpy(), mode='same')    
result_same = correlate(x.detach().numpy(), x.detach().numpy(), mode='same')

# Find peaks in the correlation result
peaks, _ = find_peaks(result, height=0.5 * np.max(result))

# Plot the original signal, the reconstructed signal, and the correlation result
plt.figure(figsize=(12, 8))
plt.subplot(3, 1, 1)
plt.plot(x.detach().numpy(), label='Original Signal')
plt.title('Original Signal')
plt.legend()
plt.subplot(3, 1, 2)
plt.plot(y.detach().numpy(), label='Reconstructed Signal')
plt.title('Reconstructed Signal')
plt.legend()
plt.subplot(3, 1, 3)
plt.plot(result, label='Correlation Result')
plt.plot(result_same, label='Autocorrelation of Original Signal', linestyle='--')
plt.plot(peaks, result[peaks], "x", label='Peaks in Correlation')
plt.title('Correlation between Original and Reconstructed Signal')
plt.legend()
plt.tight_layout()
plt.show()

# From the peaks in the correlation result, we can infer the time lag between the original and reconstructed signals. The location of the peaks indicates where the two signals are most similar, which can help us understand any temporal shifts or delays in the reconstruction process.

This first trial was done via a temporal correlation function, which is more than suboptimal, now i would like to use a spatial correlation function, as provided in utils.py

In [ ]:
from utils import analyze_plasma_profiles

In [ ]:
# Get the profile for a specific PID and time instant 
pid = "20250402.14"
t = 2.1 # time instant in seconds
dat = np.arange(360) # dummy theta values for plotting

# Look for the profile in the dataset
mask = data_module.full_data.pids == pid
print(f"Found {mask.sum()} profiles for PID {pid}")
mask_time = np.isclose(data_module.full_data.times[mask], t, atol=0.1) # Look for profiles within 0.1 seconds of the specified time
print(f"Found {mask_time.sum()} profiles for time {t} seconds")

# Get the profile and reconstruction for the first matching profile
profile = data_module.full_data.profiles[mask][mask_time][0].detach().numpy()
profile_tensor = torch.tensor(profile, dtype=torch.float32)
reconstruction = model(profile_tensor).detach().cpu().numpy()

analyze_plasma_profiles(dat, profile, reconstruction, pid=pid)

# # Get the profile and reconstruction for a random experiment
# for batch in full_dl:
#     print(batch["pid"])
#     print(batch["time"])
#     print(batch["profile"].shape)
    
#     x = batch["profile"][54]
#     y = model(x)

# profile = x.detach().numpy()
# reconstruction = y.detach().numpy()

# analyze_plasma_profiles(dat, profile, reconstruction)

Let's now try and do the analisys on the poloidal section of the machine, giving as input not the radius but an angle. 

In [ ]:
from utils import analyze_poloidal_profiles

# # One can use the pid and t from the previous cell 
# # Get the profile for a specific PID and time instant 
# pid = "20250312.114"
# t = 4.5 # time instant in seconds

# # Look for the profile in the dataset
# mask = data_module.full_data.pids == pid
# print(f"Found {mask.sum()} profiles for PID {pid}")
# mask_time = np.isclose(data_module.full_data.times[mask], t, atol=1e-14) # Look for profiles within 0.1 seconds of the specified time
# print(f"Found {mask_time.sum()} profiles for time {t} seconds")

# Get the profile and reconstruction for the first matching profile
profile = data_module.full_data.profiles[mask][mask_time][0].detach().numpy()
profile_tensor = torch.tensor(profile, dtype=torch.float32)
reconstruction = model(profile_tensor).detach().cpu().numpy()

theta = np.linspace(0, 2*np.pi, 360, endpoint=False)

analyze_poloidal_profiles(theta, profile, reconstruction)
plt.show()

In [ ]:
print(gain_list)
print(f"Available PIDs in the dataset: {data_module.full_data.pids}")
# Now get how many of the gain_list pids are in the dataset
gain_pids_in_dataset = [pid for pid in gain_list if pid in data_module.full_data.pids]
print(f"GAIN PIDs in the dataset: {gain_pids_in_dataset}")

print(f"Number of available PIDs in the dataset: {len(data_module.full_data.pids)}")
print(f"Number of PIDs in the GAIN list: {len(gain_list)}")
print(f"Number of GAIN PIDs in the dataset: {len(gain_pids_in_dataset)}")